In [4]:
!pip install -q chromadb langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 13.2 MB/s eta 0:00:0000:01


In [5]:
import os
import json
from pathlib import Path
from getpass import getpass
from typing import TypedDict, Dict, Any, List, Literal

import pandas as pd
import chromadb
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Set models and storage config

In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_PATH = "career_chroma_db"
COLLECTION_NAME = "career_multi_agent_kb"
TOP_K = 5


embed_model = SentenceTransformer(EMBED_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
USER_PROFILE = {
    "name": "Md Mohsin",
    "target_role": "AI Engineer Intern",
    "experience_level": "student",
    "education": "CSE student",
    "current_skills": [
        "Python",
        "Machine Learning",
        "Deep Learning",
        "NLP",
        "Computer Vision",
        "scikit-learn"
    ],
    "tools_and_frameworks": [
        "PyTorch",
        "Transformers",
        "Flask",
        "Django",
        "Git",
        "Pandas"
    ],
    "projects_done": [
        "Spam detection NLP project",
        "E-commerce backend API project",
        "Vision transformer fine-tuning practice"
    ],
    "cv_text": """
Md Mohsin
CSE student interested in AI and backend development.
Worked on spam detection using machine learning and Flask.
Built an e-commerce backend API using Django REST Framework.
Learned ANN, CNN, RNN and worked on transformer fine-tuning practice.
Interested in AI engineer internship opportunities.
""",
    "target_job_description": """
We are hiring an AI Engineer Intern.
Requirements include Python, data preprocessing, model evaluation, machine learning fundamentals,
deep learning, NLP or computer vision exposure, Git, SQL basics, API integration,
clear communication, experimentation mindset, and ability to build end-to-end AI projects.
Experience with LLMs, vector databases, deployment, and notebooks is a plus.
""",
    "weekly_hours_available": 15,
    "notes": "I want a fresher-friendly roadmap and project ideas that improve my internship chances."
}

### Define structured schemas

In [8]:
class UserProfile(BaseModel):
    name: str
    target_role: str
    experience_level: Literal["student", "fresher", "junior", "switcher"] = "fresher"
    education: str = ""
    current_skills: List[str] = Field(default_factory=list)
    tools_and_frameworks: List[str] = Field(default_factory=list)
    projects_done: List[str] = Field(default_factory=list)
    cv_text: str
    target_job_description: str = ""
    weekly_hours_available: int = 12
    notes: str = ""

class CVReviewOutput(BaseModel):
    strengths: List[str]
    cv_improvement_points: List[str]
    rewritten_professional_summary: str
    keyword_gaps: List[str]

class SkillsGapOutput(BaseModel):
    role_fit_summary: str
    existing_strengths: List[str]
    missing_skills: List[str]
    missing_tools: List[str]
    learning_priority_order: List[str]

In [9]:
class ProjectIdea(BaseModel):
    title: str
    goal: str
    tech_stack: List[str]
    deliverables: List[str]
    why_this_project: str

class ProjectSuggestionsOutput(BaseModel):
    recommended_projects: List[ProjectIdea]

class InterviewTopic(BaseModel):
    topic: str
    why_it_matters: str
    sample_questions: List[str]

In [10]:
class InterviewPlanOutput(BaseModel):
    priority_topics: List[InterviewTopic]
    weekly_practice_strategy: str

class WeekPlan(BaseModel):
    week: int
    focus: str
    tasks: List[str]
    deliverable: str

class RoadmapOutput(BaseModel):
    thirty_day_goal: str
    weekly_plan: List[WeekPlan]
    checkpoint_metrics: List[str]

class FinalCareerPlan(BaseModel):
    coordinator_summary: str
    cv_improvement_points: List[str]
    missing_skills: List[str]
    project_recommendations: List[ProjectIdea]
    interview_prep_roadmap: List[InterviewTopic]
    thirty_day_plan: List[WeekPlan]
    next_7_days: List[str]